# 实战案例 3：生信分析集群作业调度（离散事件仿真 + Q-Learning）

**真实场景**：WGS/RNA-seq 流水线任务到达集群，含 **CPU 轻量**与 **GPU 重计算**两类 job；调度器决定将队首任务派到 3 台异构节点，目标降低**平均等待时间**与 **SLA 超时率**。

**本 Notebook**：
1. 用 NumPy 实现简化事件仿真
2. 对比 FIFO vs **Q-Learning 调度**
3. 可视化等待时间与 SLA 违反率

与 [K8s 调度](../../02.开发-04.pipeline/pipeline-frameworks-k8s-04.工作负载与资源调度.md) 的关系：K8s 执行资源声明，本案例学习**派单策略**。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import deque

np.random.seed(42)
N_NODES = 3
# 节点类型: 0=CPU-only, 1=GPU, 2=GPU大内存
NODE_GPU = np.array([0, 1, 1])
NODE_SPEED = np.array([1.0, 1.5, 2.0])  # 相对处理速度
SLA_STEPS = 80  # 超过此等待步数记 SLA 违反

## 1. 仿真环境

In [ ]:
class ClusterSim:
    """简化集群：每步调度队首 job 到某节点，节点 busy 则 countdown。"""
    def __init__(self, horizon=500):
        self.horizon = horizon
        self.reset()

    def reset(self):
        self.t = 0
        self.queue = deque()  # (job_type, wait)
        self.node_busy = np.zeros(N_NODES)  # 剩余处理时间
        self.total_wait = 0
        self.sla_viol = 0
        self.n_done = 0
        return self._state()

    def _state(self):
        # 状态: 队首 job 类型(0/1), 队列长度, 三节点 busy 二值
        head = self.queue[0][0] if self.queue else 0
        qlen = min(len(self.queue), 10)
        busy = (self.node_busy > 0).astype(int)
        return head * 100 + qlen * 10 + busy[0]*4 + busy[1]*2 + busy[2]

    def _arrivals(self):
        if np.random.rand() < 0.35:
            jtype = 1 if np.random.rand() < 0.4 else 0  # 40% GPU job
            dur = np.random.randint(5, 15) if jtype == 0 else np.random.randint(10, 25)
            self.queue.append([jtype, dur, 0])  # type, work, wait

    def step(self, action):
        # action: 0..2 选节点; 若队列空则 noop
        self._arrivals()
        reward = 0
        if self.queue:
            job = self.queue[0]
            jtype, work, wait = job
            node = int(action) % N_NODES
            # GPU job 派到 CPU 节点则惩罚
            mismatch = (jtype == 1 and NODE_GPU[node] == 0)
            if self.node_busy[node] <= 0:
                speed = NODE_SPEED[node] * (0.5 if mismatch else 1.0)
                self.node_busy[node] = work / speed
                self.queue.popleft()
                reward = -wait - (20 if mismatch else 0)
                if wait > SLA_STEPS:
                    self.sla_viol += 1
                self.total_wait += wait
                self.n_done += 1
            else:
                reward = -1  # 节点忙仍强派
        for i in range(N_NODES):
            if self.node_busy[i] > 0:
                self.node_busy[i] -= 1
        for job in self.queue:
            job[2] += 1  # wait++
        self.t += 1
        done = self.t >= self.horizon
        return self._state(), reward, done

sim = ClusterSim()
print("初始状态 id:", sim.reset())

## 2. FIFO Baseline

In [ ]:
def run_fifo(seed=0):
    np.random.seed(seed)
    env = ClusterSim(horizon=600)
    env.reset()
    done = False
    while not done:
        # FIFO: 选第一个空闲节点
        a = int(np.argmin(env.node_busy))
        _, _, done = env.step(a)
    avg_wait = env.total_wait / max(env.n_done, 1)
    return avg_wait, env.sla_viol, env.n_done

fifo_wait, fifo_sla, fifo_n = run_fifo()
print(f"FIFO: avg_wait={fifo_wait:.1f}, SLA_viol={fifo_sla}, jobs={fifo_n}")

## 3. Q-Learning 调度器

In [ ]:
def run_qlearning(episodes=120, alpha=0.2, gamma=0.95, eps=0.2):
    n_states = 300
    Q = np.zeros((n_states, N_NODES))
    logs = []
    for ep in range(episodes):
        env = ClusterSim(horizon=400)
        s = env.reset()
        done = False
        while not done:
            if np.random.rand() < eps:
                a = np.random.randint(N_NODES)
            else:
                a = Q[min(s, n_states-1)].argmax()
            s2, r, done = env.step(a)
            si = min(s, n_states-1); s2i = min(s2, n_states-1)
            Q[si, a] += alpha * (r + gamma * Q[s2i].max() - Q[si, a])
            s = s2
        avg_w = env.total_wait / max(env.n_done, 1)
        logs.append((avg_w, env.sla_viol))
        eps = max(0.05, eps * 0.995)
    return Q, logs

Q, logs = run_qlearning()
rl_wait, rl_sla = logs[-1]
print(f"Q-Learning (末轮): avg_wait={rl_wait:.1f}, SLA_viol={rl_sla}")

In [ ]:
waits = [x[0] for x in logs]
plt.figure(figsize=(8,4))
plt.plot(waits, label="Q-Learning avg wait")
plt.axhline(fifo_wait, color="r", ls="--", label=f"FIFO baseline ({fifo_wait:.1f})")
plt.xlabel("Episode"); plt.ylabel("Avg wait"); plt.legend(); plt.title("调度策略学习曲线")
plt.tight_layout(); plt.show()
print(f"相对 FIFO 等待时间变化: {(rl_wait - fifo_wait)/fifo_wait*100:.1f}%")

## 4. 落地思考

| 难点 | 方案 | 优势 | 局限 |
|------|------|------|------|
| 状态空间大 | 表格 Q + 状态聚合 | 简单 | 泛化差 |
| 非平稳到达 | 定期重训 + 滑动窗口 | 适应流量变化 | 需离线 pipeline |
| 与 K8s 集成 | scheduler extender / 二次调度 | 不改 kubelet | 延迟与一致性 |

生产建议：**仿真 trace 回放 → RL 策略 → shadow 对比 p99 等待时间 → 小流量上线**。